In [4]:
import os
import cv2
from ultralytics import YOLO

# =========================================================
# INPUT AND OUTPUT PATHS
# =========================================================
image_folder = r"C:\Users\Shivani Agarwal\Downloads\Garv_minor2\bdd100k\bdd100k\images\10k\trainA"
label_folder = r"C:\Users\Shivani Agarwal\Downloads\Garv_minor2\bdd100k\bdd100k\images\10k\label\test"

# Create output label folder
os.makedirs(label_folder, exist_ok=True)

# =========================================================
# LOAD PRETRAINED MODEL
# =========================================================
model = YOLO("yolo11n.pt")

# =========================================================
# CLASS MAPPING
# COCO classes in pretrained YOLO:
# 0 = person
# 1 = bicycle
# 2 = car
# 3 = motorcycle
# 5 = bus
# 7 = truck
#
# New custom IDs:
# 0 = car
# 1 = bus
# 2 = truck
# 3 = motorcycle
# 4 = bicycle
# 5 = person
# =========================================================
allowed_classes = {
    2: 0,   # car
    5: 1,   # bus
    7: 2,   # truck
    3: 3,   # motorcycle
    1: 4,   # bicycle
    0: 5    # person
}

# =========================================================
# SETTINGS
# =========================================================
valid_ext = (".jpg", ".jpeg", ".png")
conf_threshold = 0.25

# =========================================================
# PROCESS ALL IMAGES IN trainB
# =========================================================
if not os.path.exists(image_folder):
    print(f"[WARNING] Folder not found: {image_folder}")
else:
    print(f"\nProcessing images from: {image_folder}")
    image_files = [f for f in os.listdir(image_folder) if f.lower().endswith(valid_ext)]

    for idx, img_name in enumerate(image_files, start=1):
        img_path = os.path.join(image_folder, img_name)
        img = cv2.imread(img_path)

        if img is None:
            print(f"[WARNING] Could not read image: {img_path}")
            continue

        h, w = img.shape[:2]

        # Run detection
        results = model(img_path, conf=conf_threshold, verbose=False)

        label_lines = []

        for result in results:
            if result.boxes is None:
                continue

            for box in result.boxes:
                cls_id = int(box.cls[0].item())

                if cls_id not in allowed_classes:
                    continue

                x1, y1, x2, y2 = box.xyxy[0].tolist()

                # Convert to YOLO format
                x_center = ((x1 + x2) / 2) / w
                y_center = ((y1 + y2) / 2) / h
                bw = (x2 - x1) / w
                bh = (y2 - y1) / h

                # Clamp values for safety
                x_center = max(0.0, min(1.0, x_center))
                y_center = max(0.0, min(1.0, y_center))
                bw = max(0.0, min(1.0, bw))
                bh = max(0.0, min(1.0, bh))

                new_class_id = allowed_classes[cls_id]

                label_lines.append(
                    f"{new_class_id} {x_center:.6f} {y_center:.6f} {bw:.6f} {bh:.6f}"
                )

        # Save .txt file
        txt_name = os.path.splitext(img_name)[0] + ".txt"
        txt_path = os.path.join(label_folder, txt_name)

        with open(txt_path, "w") as f:
            f.write("\n".join(label_lines))

        print(f"[{idx}/{len(image_files)}] Saved: {txt_path}")

print("\nDone. Labels saved in:")
print(label_folder)


Processing images from: C:\Users\Shivani Agarwal\Downloads\Garv_minor2\bdd100k\bdd100k\images\10k\trainA
[1/1698] Saved: C:\Users\Shivani Agarwal\Downloads\Garv_minor2\bdd100k\bdd100k\images\10k\label\test\ac517380-00000000.txt
[2/1698] Saved: C:\Users\Shivani Agarwal\Downloads\Garv_minor2\bdd100k\bdd100k\images\10k\label\test\ac56c836-bdabca21.txt
[3/1698] Saved: C:\Users\Shivani Agarwal\Downloads\Garv_minor2\bdd100k\bdd100k\images\10k\label\test\ac6d4f42-00000000.txt
[4/1698] Saved: C:\Users\Shivani Agarwal\Downloads\Garv_minor2\bdd100k\bdd100k\images\10k\label\test\ac6e638d-7c84846d.txt
[5/1698] Saved: C:\Users\Shivani Agarwal\Downloads\Garv_minor2\bdd100k\bdd100k\images\10k\label\test\ac73d367-0cb39ad0.txt
[6/1698] Saved: C:\Users\Shivani Agarwal\Downloads\Garv_minor2\bdd100k\bdd100k\images\10k\label\test\ac985232-00000000.txt
[7/1698] Saved: C:\Users\Shivani Agarwal\Downloads\Garv_minor2\bdd100k\bdd100k\images\10k\label\test\ac9be3fe-790d1f8e.txt
[8/1698] Saved: C:\Users\Shivani 

In [1]:
import os
import cv2
from ultralytics import YOLO

# =========================================================
# PATHS
# =========================================================
images_base = r"C:\Users\Shivani Agarwal\Downloads\Garv_minor2\bdd100k\bdd100k\images\10k"
labels_base = os.path.join(images_base, "label")

# Create label base folder
os.makedirs(labels_base, exist_ok=True)

# =========================================================
# LOAD PRETRAINED MODEL
# =========================================================
model = YOLO("yolo11n.pt")

# =========================================================
# CLASS MAPPING
# COCO classes in pretrained YOLO:
# 0 = person
# 1 = bicycle
# 2 = car
# 3 = motorcycle
# 5 = bus
# 7 = truck
#
# New custom IDs:
# 0 = car
# 1 = bus
# 2 = truck
# 3 = motorcycle
# 4 = bicycle
# 5 = person
# =========================================================
allowed_classes = {
    2: 0,   # car
    5: 1,   # bus
    7: 2,   # truck
    3: 3,   # motorcycle
    1: 4,   # bicycle
    0: 5    # person
}

# =========================================================
# SETTINGS
# =========================================================
splits = ["train", "val", "test"]
valid_ext = (".jpg", ".jpeg", ".png")
conf_threshold = 0.25

# =========================================================
# PROCESS EACH SPLIT
# =========================================================
for split in splits:
    image_folder = os.path.join(images_base, split)
    label_folder = os.path.join(labels_base, split)

    os.makedirs(label_folder, exist_ok=True)

    if not os.path.exists(image_folder):
        print(f"[WARNING] Folder not found: {image_folder}")
        continue

    print(f"\nProcessing {split} folder...")
    image_files = [f for f in os.listdir(image_folder) if f.lower().endswith(valid_ext)]

    for idx, img_name in enumerate(image_files, start=1):
        img_path = os.path.join(image_folder, img_name)
        img = cv2.imread(img_path)

        if img is None:
            print(f"[WARNING] Could not read image: {img_path}")
            continue

        h, w = img.shape[:2]

        # Run detection
        results = model(img_path, conf=conf_threshold, verbose=False)

        label_lines = []

        for result in results:
            if result.boxes is None:
                continue

            for box in result.boxes:
                cls_id = int(box.cls[0].item())

                if cls_id not in allowed_classes:
                    continue

                x1, y1, x2, y2 = box.xyxy[0].tolist()

                # Convert to YOLO format
                x_center = ((x1 + x2) / 2) / w
                y_center = ((y1 + y2) / 2) / h
                bw = (x2 - x1) / w
                bh = (y2 - y1) / h

                # Safety clamp
                x_center = max(0.0, min(1.0, x_center))
                y_center = max(0.0, min(1.0, y_center))
                bw = max(0.0, min(1.0, bw))
                bh = max(0.0, min(1.0, bh))

                new_class_id = allowed_classes[cls_id]

                label_lines.append(
                    f"{new_class_id} {x_center:.6f} {y_center:.6f} {bw:.6f} {bh:.6f}"
                )

        # Save label file
        txt_name = os.path.splitext(img_name)[0] + ".txt"
        txt_path = os.path.join(label_folder, txt_name)

        with open(txt_path, "w") as f:
            f.write("\n".join(label_lines))

        print(f"[{split}] {idx}/{len(image_files)} Saved: {txt_path}")

print("\nDone. Labels generated inside:")
print(labels_base)


Processing train folder...
[train] 1/7000 Saved: C:\Users\Shivani Agarwal\Downloads\Garv_minor2\bdd100k\bdd100k\images\10k\label\train\0004a4c0-d4dff0ad.txt
[train] 2/7000 Saved: C:\Users\Shivani Agarwal\Downloads\Garv_minor2\bdd100k\bdd100k\images\10k\label\train\00054602-3bf57337.txt
[train] 3/7000 Saved: C:\Users\Shivani Agarwal\Downloads\Garv_minor2\bdd100k\bdd100k\images\10k\label\train\00067cfb-e535423e.txt
[train] 4/7000 Saved: C:\Users\Shivani Agarwal\Downloads\Garv_minor2\bdd100k\bdd100k\images\10k\label\train\00091078-59817bb0.txt
[train] 5/7000 Saved: C:\Users\Shivani Agarwal\Downloads\Garv_minor2\bdd100k\bdd100k\images\10k\label\train\0010bf16-a457685b.txt
[train] 6/7000 Saved: C:\Users\Shivani Agarwal\Downloads\Garv_minor2\bdd100k\bdd100k\images\10k\label\train\001b428f-059bac33.txt
[train] 7/7000 Saved: C:\Users\Shivani Agarwal\Downloads\Garv_minor2\bdd100k\bdd100k\images\10k\label\train\001c2a14-c7138401.txt
[train] 8/7000 Saved: C:\Users\Shivani Agarwal\Downloads\Garv_